## **Autograd**

In [18]:
# two cases -> this could be a constant input, or an output composed of other spyders.
class Spyder:
    def __init__(self, data, parents = None):
        self.data = data
        self.grad = 0
        self.parents = parents if parents else []
        self._backward = lambda: None

    def __mul__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)
        out = Spyder(self.data * other.data, parents = [self, other])

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)
        out = Spyder(self.data + other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)
        out = Spyder(self.data - other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad
            other.grad -= out.grad

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float values supported"
        out = Spyder(self.data ** other, parents = [self,])

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __repr__(self):
        return f"Spyder(data: {self.data}, grad: {self.grad})"

    def retrace(self):
        self.clean_web()
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited: 
                visited.add(v)
                for parent in v.parents:
                    build_topo(parent)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1

        for v in reversed(topo):
            v._backward()

    def clean_web(self):
        visited = set()
        def visit_children(v):
            if v not in visited: 
                visited.add(v)
                v.grad = 0

                for parent in v.parents:
                    visit_children(parent)
        visit_children(self)

In [19]:
x = Spyder(2)
a = x ** 3

In [20]:
a.retrace()
x.grad

12

In [28]:
# two cases -> this could be a constant input, or an output composed of other spyders.
import numpy as np
class Spyder:
    def __init__(self, data: np.ndarray, parents = None):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self.parents = parents if parents else []
        self._backward = lambda: None

    def __repr__(self):
        return f"Spyder(data = {self.data}, grad = {self.grad}, parents = {self.parents})"

    def __mul__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)

        assert self.data.shape == other.data.shape, "not supporting broadcasting for now"
        out = Spyder(self.data * other.data, parents=[self, other])

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __matmul__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)

        assert self.data.shape[1] == other.data.shape[0], "matrixes must be in shape (m, n), (n, p) for matmul"
        out = Spyder(self.data @ other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad

        out._backward = _backward
        return out

    def sum(self):
        out = Spyder(np.sum(self.data), parents = [self,])

        def _backward():
            self.grad += np.ones_like(self.data) * out.grad

        out._backward = _backward
        return out

    def mean(self):
        out = Spyder(np.mean(self.data), parents = [self])

        def _backward():
            self.grad += (
                np.ones_like(self.data) *
                out.grad /
                self.data.size
            )

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert (isinstance(other, (int, float))), "only supporting int/float powers for now"

        out = Spyder(self.data ** other, parents = [self]) 

        def _backward():
            self.grad += (other * (self.data) ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)

        assert (
            self.data.shape == other.data.shape or # (m, n) + (m, n)
                (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) + (n,)
                ), "not supporting broadcasting for now"
        
        out = Spyder(self.data + other.data, parents=[self, other])

        def _backward():

            if self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad += np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad += np.sum(out.grad, axis = 0)

        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)

        assert (
            self.data.shape == other.data.shape or # (m, n) + (m, n)
                (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) + (n,)
                ), "not supporting broadcasting for now"
        
        out = Spyder(self.data - other.data, parents=[self, other])

        def _backward():

            if self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad -= np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad -= np.sum(out.grad, axis = 0)

        out._backward = _backward
        return out

    def __neg__(self):
        out = Spyder(-self.data, parents = [self])
        def _backward():
            self.grad -= out.grad
        out._backward = _backward
        return out

    def __rsub__(self, other):
        assert isinstance(other, (int, float))

        other = Spyder(
            np.full(self.data.shape, other)
        )

        return other - self

    def __getitem__(self, idx):
        out = Spyder(self.data[idx], parents = [self])

        def _backward():
            self.grad[idx] += out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Spyder(np.maximum(0, self.data), parents = [self])

        def _backward():
            self.grad += (self.data > 0) * out.grad

        out._backward = _backward
        return out

    def sigmoid(self):
        out = Spyder(1 / (1 + np.exp(-self.data)), parents = [self])

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad

        out._backward = _backward
        return out

    def log(self):
        out = Spyder(np.log(self.data), parents = [self])

        def _backward():
            self.grad += (1 / self.data) * out.grad

        out._backward = _backward
        return out
    
    def retrace(self):
        self.clean_webs()
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for parent in v.parents:
                    build_topo(parent)
                topo.append(v)
        build_topo(self)

        self.grad = np.ones_like(self.data)
        for v in reversed(topo):
            v._backward()

    def clean_webs(self):
        visited = set()
        def visit_parents(v):
            if v not in visited:
                visited.add(v)
                v.grad = np.zeros_like(v.data)
                for parent in v.parents:
                    visit_parents(parent)
        visit_parents(self)

In [30]:
import numpy as np
class Web:
    def __init__(self, layer_size, learning_rate = 0.001, intialization_strength = 0.01, random_state = 11, epochs = 1000):

        """
        x, y -> np.ndarray or spyder
        layer_size -> (input features, layer 1 neurons, layer 2 neurons, ..., outputs)
        """
        self.rng = np.random.default_rng(random_state)
        self.layer_size = layer_size
        self.learning_rate = learning_rate
        self.intialization_strength = intialization_strength
        self.epochs = epochs

    def spin(self, x, y):

        x = x if isinstance(x, Spyder) else Spyder(x)
        y = y if isinstance(y, Spyder) else Spyder(y)

        self.weights = []
        self.biases = []

        for n in range(len(self.layer_size) - 1):
        
            in_features = self.layer_size[n]
            out_features = self.layer_size[n + 1]
        
            self.weights.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (in_features, out_features))))
            self.biases.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (out_features,))))

        for epoch in range(self.epochs):
            inp = x
            for w, b in zip(self.weights[:-1], self.biases[:-1]):
                inp = (inp @ w + b).relu()

            pred = inp @ self.weights[-1] + self.biases[-1]
            error = ((pred - y) ** 2).mean()
            
            error.retrace()

            for w, b in zip(self.weights, self.biases):
                w.data -= self.learning_rate * w.grad
                b.data -= self.learning_rate * b.grad

            if epoch % 50 == 0:
                print(f"epoch {epoch}, loss = {error.data}")

    def predict(self, x):
        inp = x if isinstance(x, Spyder) else Spyder(x)
        for w, b in zip(self.weights[:-1], self.biases[:-1]):
            inp = (inp @ w + b).relu()
        pred = inp @ self.weights[-1] + self.biases[-1]
        return pred

In [31]:
np.array([2, 3, 4]).size

3

In [32]:
x = Spyder(np.array([[3.0]]))
w = Spyder(np.array([[2.0]]))

y = x @ w
loss = y.sum()

loss.retrace()

print(w.grad)

[[3.]]


In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [34]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [35]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [36]:
x_train = transformations.fit_transform(x_train)

In [37]:
x_test = transformations.transform(x_test)

In [38]:
web = Web([x_train.shape[1],32, 16, 1], learning_rate = 0.05, intialization_strength=0.01, epochs = 1500)

In [39]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.to_numpy().reshape(-1, 1)
)

In [40]:
web.spin(x_train[:5000], y_train_scaled[:5000])

epoch 0, loss = 0.9903153545974539
epoch 50, loss = 0.9901224047944058
epoch 100, loss = 0.9901176155578978
epoch 150, loss = 0.9901126497029287
epoch 200, loss = 0.9901067905092604
epoch 250, loss = 0.9900992633700533
epoch 300, loss = 0.9900890228042717
epoch 350, loss = 0.990074469011847
epoch 400, loss = 0.9900529533257215
epoch 450, loss = 0.9900198607935865
epoch 500, loss = 0.9899664364748465
epoch 550, loss = 0.9898750973831524
epoch 600, loss = 0.9897056882408503
epoch 650, loss = 0.9893307795536295
epoch 700, loss = 0.9884982560916734
epoch 750, loss = 0.9862716444963658
epoch 800, loss = 0.9789632340933538
epoch 850, loss = 0.9475913874621398
epoch 900, loss = 0.7571758235760839
epoch 950, loss = 0.18679287902667988
epoch 1000, loss = 0.24534535852514672
epoch 1050, loss = 0.17915977706742797
epoch 1100, loss = 0.14236126251050432
epoch 1150, loss = 0.12059725693660757
epoch 1200, loss = 0.10636877145585154
epoch 1250, loss = 0.09514690260880251
epoch 1300, loss = 0.08614254

In [41]:
pred = web.predict(x_test)

In [42]:
train_pred = web.predict(x_train)
train_mse = np.mean((train_pred.data - y_train_scaled.reshape(-1,1))**2)

In [43]:
train_mse

np.float64(0.07217249108844467)

In [44]:
mse = np.mean((pred.data - y_test_scaled)**2)

In [45]:
mse

np.float64(0.07121071433434586)

In [46]:
import gc

print(len(gc.get_objects()))

241987


In [93]:
# current architecture: just a go through
import numpy as np # autograd 
import numbers 
class Spyder:
    def __init__(self, data: np.ndarray, parents = None):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self.parents = parents if parents else []
        self._backward = lambda: None

    def _coerce(self, other):
        """function to coerce other into a spyder"""
        match other:
            case Spyder(): return other
            case isinstance(other, (numbers.Real, np.generic)): return Spyder(other) 
            case _: raise TypeError(
                f"Received object of type {type(other)}, was unable to coerce to Spyder object"
                f"\nOnly supporting scalars, np.ndarrays and other spyders!"
                )

    def __repr__(self):
        return f"Spyder(data = {self.data}, grad = {self.grad}, parents = {self.parents})"

    def __matmul__(self, other):
        other = self._coerce(other)

        try:
            assert self.data.shape[1] == other.data.shape[0], "matrixes must be in shape (m, n), (n, p) for matmul"
        except IndexError:
            raise ValueError("Sorry bro i was too lazy to implement any other broadcasting type :(\n"
                            f"So i got a matrices of shapes {self.data.shape} and {other.data.shape}"
                            "it is not in required (m, n), (n, p) form so i cannot calculate sorry :((\n"
                            "just use tinygrad or micrograd bro i so sorry pls forgive me")
        out = Spyder(self.data @ other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad

        out._backward = _backward
        return out

    def sum(self):
        out = Spyder(np.sum(self.data), parents = [self,])

        def _backward():
            self.grad += np.ones_like(self.data) * out.grad

        out._backward = _backward
        return out

    def mean(self):
        out = Spyder(np.mean(self.data), parents = [self])

        def _backward():
            self.grad += (
                np.ones_like(self.data) *
                out.grad /
                self.data.size
            )

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert (isinstance(other, (numbers.Real, np.generic))), "only supporting int/float powers for now"

        out = Spyder(self.data ** other, parents = [self]) 

        def _backward():
            self.grad += (other * (self.data) ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):
        if isinstance(other, (numbers.Real, np.generic)): # now we know that this isnt a spyder with a scalar value, its just a constant
            out = Spyder(self.data + other, parents=[self])

            def _backward():
                self.grad += out.grad

            out._backward = _backward

            return out
        
        other = self._coerce(other)

        assert (
            self.data.shape == other.data.shape or # (m, n) + (m, n)
                (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) + (n,)
                ), "not supporting broadcasting for now"
        
        out = Spyder(self.data + other.data, parents=[self, other])

        def _backward():

            if self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad += np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad += np.sum(out.grad, axis = 0)

        out._backward = _backward
        return out

    def __sub__(self, other):
        if isinstance(other, (numbers.Real, np.generic)): # now we know that this isnt a spyder with a scalar value, its just a constant
            out = Spyder(self.data - other, parents=[self]) 
    
            def _backward():
                self.grad += out.grad 
                out._backward = _backward
        
            return out
                
        other = self._coerce(other)

        assert (
            self.data.shape == other.data.shape or # (m, n) - (m, n)
                (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) - (n,)
                ), "not supporting broadcasting for now"
        
        out = Spyder(self.data - other.data, parents=[self, other])

        def _backward():

            if self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad -= np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad -= np.sum(out.grad, axis = 0)

        out._backward = _backward
        return out

    def __mul__(self, other):
        if isinstance(other, (numbers.Real, np.generic)): # now we know that this isnt a spyder with a scalar value, its just a constant
            out = Spyder(self.data * other, parents=[self]) 
            
            def _backward():
                self.grad += other * out.grad
            
            out._backward = _backward
            
            return out
                    
        other = self._coerce(other)
    
        assert (
                self.data.shape == other.data.shape or # (m, n) * (m, n)
                    (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) * (n,)
                    ), "not supporting broadcasting for now" 
        out = Spyder(self.data * other.data, parents=[self, other])
    
        def _backward():
            if self.data.shape == other.data.shape:
                self.grad += other.data * out.grad
                other.grad += self.data * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += other.data * out.grad 
                other.grad += np.sum(self.data * out.grad, axis = 0) 
    
        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = self._coerce(other)

        return self * (other ** -1) # lol this is funny for some reason :sob:

    def __neg__(self):
        out = Spyder(-self.data, parents = [self])
        def _backward():
            self.grad -= out.grad
        out._backward = _backward
        return out

    def __rsub__(self, other):
        return -self + other 

    def __radd__(self, other):
        return self + other # addition is commutative 

    def __rmul__(self, other):
        return self * other 

    def __rtruediv__(self, other):
        return other * (self ** -1) 

    def __getitem__(self, idx):
        out = Spyder(self.data[idx], parents = [self])

        def _backward():
            self.grad[idx] += out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Spyder(np.maximum(0, self.data), parents = [self])

        def _backward():
            self.grad += (self.data > 0) * out.grad

        out._backward = _backward
        return out

    def sigmoid(self):
        x = self.data

        out_data = np.empty_like(x)

        positive = x >= 0 # prepare boolean mask
        negative = ~positive

        out_data[positive] = 1 / (1 + np.exp(-x[positive])) 
        out_data[negative] = np.exp(x[negative]) / (1 + np.exp(x[negative]))

        out = Spyder(out_data, parents = [self])

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad

        out._backward = _backward
        return out

    def log(self):
        x = np.clip(self.data, 1e-8, None)
        out = Spyder(np.log(x), parents = [self]) # clip x value to make sure log doesnt go boom

        def _backward():
            self.grad += (self.data >= 1e-8) / x * out.grad 

        out._backward = _backward
        return out
    
    def retrace(self):
        self.clean_webs()
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for parent in v.parents:
                    build_topo(parent)
                topo.append(v)
        build_topo(self)

        self.grad = np.ones_like(self.data)
        for v in reversed(topo): 
            v._backward()

    def clean_webs(self):
        visited = set()
        def visit_parents(v):
            if v not in visited:
                visited.add(v)
                v.grad = np.zeros_like(v.data)
                for parent in v.parents:
                    visit_parents(parent)
        visit_parents(self)

In [94]:
import numpy as np
class Web: 
    def __init__(self, layer_size, learning_rate = 0.001, intialization_strength = 0.01, random_state = 11, epochs = 1000):

        """
        x, y -> np.ndarray or spyder
        layer_size -> (input features, layer 1 neurons, layer 2 neurons, ..., outputs)
        """
        self.rng = np.random.default_rng(random_state)
        self.layer_size = layer_size
        self.learning_rate = learning_rate
        self.intialization_strength = intialization_strength
        self.epochs = epochs

    def spin(self, x, y):

        x = x if isinstance(x, Spyder) else Spyder(x)
        y = y if isinstance(y, Spyder) else Spyder(y)

        self.weights = []
        self.biases = []

        for n in range(len(self.layer_size) - 1):
        
            in_features = self.layer_size[n]
            out_features = self.layer_size[n + 1]
        
            self.weights.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (in_features, out_features))))
            self.biases.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (out_features,))))

        for epoch in range(self.epochs):
            inp = x
            for w, b in zip(self.weights[:-1], self.biases[:-1]):
                inp = (inp @ w + b).relu()

            pred = inp @ self.weights[-1] + self.biases[-1]
            error = ((pred - y) ** 2).mean() 
            
            error.retrace()

            for w, b in zip(self.weights, self.biases):
                w.data -= self.learning_rate * w.grad
                b.data -= self.learning_rate * b.grad

            if epoch % 50 == 0:
                print(f"epoch {epoch}, loss = {error.data}")

    def predict(self, x):
        inp = x if isinstance(x, Spyder) else Spyder(x)
        for w, b in zip(self.weights[:-1], self.biases[:-1]):
            inp = (inp @ w + b).relu()
        pred = inp @ self.weights[-1] + self.biases[-1]
        return pred

In [95]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("job_salary_prediction_dataset.csv") # a dataset with like 200k records

In [96]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [97]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")), # scaler
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]), # encoding
])

In [98]:
x_train = transformations.fit_transform(x_train)
x_test = transformations.transform(x_test)

In [99]:
web = Web([x_train.shape[1],32, 16, 1], learning_rate = 0.05, intialization_strength=0.01, epochs = 1500) # my beautiful creation

In [100]:
y_scaler = StandardScaler() 

y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
    )

y_test_scaled = y_scaler.transform(
    y_test.to_numpy().reshape(-1, 1)
)

In [101]:
web.spin(x_train[:5000], y_train_scaled[:5000])

epoch 0, loss = 0.9903153545974539
epoch 50, loss = 0.9901224047944058
epoch 100, loss = 0.9901176155578978
epoch 150, loss = 0.9901126497029287
epoch 200, loss = 0.9901067905092604
epoch 250, loss = 0.9900992633700533
epoch 300, loss = 0.9900890228042717
epoch 350, loss = 0.990074469011847
epoch 400, loss = 0.9900529533257215
epoch 450, loss = 0.9900198607935865
epoch 500, loss = 0.9899664364748465
epoch 550, loss = 0.9898750973831524
epoch 600, loss = 0.9897056882408503
epoch 650, loss = 0.9893307795536295
epoch 700, loss = 0.9884982560916734
epoch 750, loss = 0.9862716444963658
epoch 800, loss = 0.9789632340933538
epoch 850, loss = 0.9475913874621398
epoch 900, loss = 0.7571758235760839
epoch 950, loss = 0.18679287902667988
epoch 1000, loss = 0.24534535852514672
epoch 1050, loss = 0.17915977706742797
epoch 1100, loss = 0.14236126251050432
epoch 1150, loss = 0.12059725693660757
epoch 1200, loss = 0.10636877145585154
epoch 1250, loss = 0.09514690260880251
epoch 1300, loss = 0.08614254

In [102]:
pred = web.predict(x_test)

In [103]:
pred = y_scaler.inverse_transform(pred.data)
mean_absolute_error(y_test, pred)

7634.587180411287

In [199]:
import numpy as np
class Web: 
    def __init__(self, layer_size, learning_rate = 0.001, intialization_strength = 0.01, threshold = 0.5, random_state = 11, epochs = 1000):

        """
        x, y -> np.ndarray or spyder
        layer_size -> (input features, layer 1 neurons, layer 2 neurons, ..., outputs)
        """
        self.rng = np.random.default_rng(random_state)
        self.layer_size = layer_size
        self.learning_rate = learning_rate
        self.intialization_strength = intialization_strength
        self.epochs = epochs
        self.threshold = threshold

    def spin(self, x, y):

        x = x if isinstance(x, Spyder) else Spyder(x)
        y = y if isinstance(y, Spyder) else Spyder(y)

        self.weights = []
        self.biases = []

        for n in range(len(self.layer_size) - 1):
        
            in_features = self.layer_size[n]
            out_features = self.layer_size[n + 1]
        
            self.weights.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (in_features, out_features))))
            self.biases.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (out_features,))))

        for epoch in range(self.epochs):
            inp = x
            for w, b in zip(self.weights[:-1], self.biases[:-1]):
                inp = (inp @ w + b).sigmoid()

            pred = (inp @ self.weights[-1] + self.biases[-1]).sigmoid()
            error = (-(y * pred.log() + (1 - y) * (1 - pred).log())).mean() 
            
            error.retrace()

            for w, b in zip(self.weights, self.biases):
                w.data -= self.learning_rate * w.grad
                b.data -= self.learning_rate * b.grad

            if epoch % 50 == 0:
                print(f"epoch {epoch}, loss = {error.data}")

    def predict_proba(self, x):
        inp = x if isinstance(x, Spyder) else Spyder(x)
        for w, b in zip(self.weights[:-1], self.biases[:-1]):
            inp = (inp @ w + b).sigmoid()
        pred = inp @ self.weights[-1] + self.biases[-1]
        return pred

    def predict(self, x):
        return ((self.predict_proba(x)).sigmoid().data >= self.threshold)

In [200]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

In [201]:
x, y = load_breast_cancer(return_X_y=True, as_frame=True)

In [202]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

In [203]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size = 0.3,
    random_state = 11
)

In [204]:
x_train_scaled = x_scaler.fit_transform(x_train.to_numpy())
x_test_scaled = x_scaler.transform(x_test.to_numpy())

In [286]:
web = Web([x_train_scaled.shape[1], 20, 10, 5, 1], 1, intialization_strength=0.5, epochs = 2000)

In [287]:
web.spin(x_train_scaled, y_train.to_numpy().reshape(-1, 1))

epoch 0, loss = 0.7062516049102819
epoch 50, loss = 0.20518754759729085
epoch 100, loss = 0.07711035296000712
epoch 150, loss = 0.053038229110371524
epoch 200, loss = 0.03997321631886153
epoch 250, loss = 0.030566882391192176
epoch 300, loss = 0.024366350764342178
epoch 350, loss = 0.01981997399415064
epoch 400, loss = 0.01571925652872627
epoch 450, loss = 0.012079205300823764
epoch 500, loss = 0.009259008424297019
epoch 550, loss = 0.007250588672852939
epoch 600, loss = 0.005834975584254
epoch 650, loss = 0.004816516470533415
epoch 700, loss = 0.004063322373718228
epoch 750, loss = 0.0034910732697766827
epoch 800, loss = 0.0030455853558719772
epoch 850, loss = 0.00269128294100062
epoch 900, loss = 0.0024042021858870152
epoch 950, loss = 0.002167783900622917
epoch 1000, loss = 0.001970308157670181
epoch 1050, loss = 0.001803295978130452
epoch 1100, loss = 0.001660489762505682
epoch 1150, loss = 0.001537187501447055
epoch 1200, loss = 0.001429798206078227
epoch 1250, loss = 0.0013355388

In [288]:
pred = web.predict(x_test_scaled)

In [289]:
from sklearn.metrics import classification_report

In [290]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.93      0.93      0.93        61
           1       0.96      0.96      0.96       110

    accuracy                           0.95       171
   macro avg       0.95      0.95      0.95       171
weighted avg       0.95      0.95      0.95       171

